# Previous Loans – Feature Engineering

Step 1 of preprocessing for `previous_application.csv`: every earlier
application each applicant made to Home Credit (one row per previous
application, `SK_ID_PREV`), whether it was approved, refused or cancelled,
and the amounts asked for and granted.

**Outputs** (`data/Model-Training/Staging/`)
- `prev_loan_data_processed.parquet` — the cleaned table, one row per
  previous application. The credit card, POS and repayment notebooks read
  it, which is why the Airflow DAG runs this notebook before them.
- `prev_loans_aggregate_final_processed.parquet` — one row per applicant
  with the summary features for the model.

**Sections**
1. Settings
2. Load, keeping only applicants in the training data
3. Turn placeholder values (`365243`, `XNA`) into missing
4. Ratio features
5. Save the cleaned loan-level table
6. Split into refused and not-refused applications
7. Helper functions
8. Summaries over all previous applications (all time + recent periods)
9. Summaries over refused applications only
10. Combine the two
11. Quality checks
12. Save
13. Free memory

Run by the `01_preprocessing` Airflow DAG (task `task_prevloans`), or by hand.

## 1. Imports & Configuration

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker,
otherwise this project's folder on your PC.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_APP_TRAIN     = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_PREV_LOANS    = os.path.join(DATA_ROOT, "Input", "previous_application.csv")
PATH_OUT_ENRICHED  = os.path.join(DATA_ROOT, "Model-Training", "Staging",
                                   "prev_loan_data_processed.parquet")
PATH_OUT_AGG       = os.path.join(DATA_ROOT, "Model-Training", "Staging",
                                   "prev_loans_aggregate_final_processed.parquet")

## 2. Load Data

Only previous applications from applicants in `application_train.csv` are
kept. Just the applicant ID is read from that file — not `TARGET` — so the
outcome being predicted can't leak into these features.

In [2]:
# Applicant IDs only. TARGET is deliberately not read, so it can't end up in
# the saved files (that would leak the answer into the features).
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR'])
base = base.astype({'SK_ID_CURR': 'string'})

PrevLoans = pd.read_csv(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_CURR': 'string', 'SK_ID_PREV': 'string'})

# Keep only applicants that are in the training data
PrevLoans = PrevLoans.merge(base, on='SK_ID_CURR', how='inner')

# Sort by applicant, most recent decision first (sorted after the merge,
# because a merge doesn't keep row order)
PrevLoans.sort_values(
    by=['SK_ID_CURR', 'DAYS_DECISION'],
    ascending=[True, False],
    inplace=True,
    ignore_index=True,
)

print(f"PrevLoans shape: {PrevLoans.shape}")

PrevLoans shape: (1413701, 37)


## 3. Turn Placeholder Values into Missing

### 3a. The number `365243`

`365243` days (about 1,000 years) means "not applicable" in several
`DAYS_*` columns (e.g. the first-due or termination date of a loan that
never started). Every numeric column is checked and each `365243` becomes
missing (`NaN`).

In [ ]:
SENTINEL = 365243
numeric_cols = PrevLoans.select_dtypes(include='number').columns
sentinel_cols = [col for col in numeric_cols if (PrevLoans[col] == SENTINEL).any()]
print(f"Columns containing sentinel {SENTINEL}: {sentinel_cols}")

for col in sentinel_cols:
    PrevLoans.loc[PrevLoans[col] == SENTINEL, col] = np.nan

print("Sentinel values replaced with NaN.")

### 3b. The text `'XNA'`

`XNA` ("not available") marks unknown values, e.g. in `NAME_PAYMENT_TYPE`.
Every text column is checked and `XNA` becomes missing.

In [ ]:
object_cols = PrevLoans.select_dtypes(include='object').columns
xna_cols = [col for col in object_cols if (PrevLoans[col] == 'XNA').any()]
print(f"Columns containing 'XNA': {xna_cols}")

for col in xna_cols:
    PrevLoans.loc[PrevLoans[col] == 'XNA', col] = np.nan

print("XNA values replaced with NaN.")

## 4. Ratio Features

- **`CREDIT_TO_APPL`** — amount granted ÷ amount asked for. Below 1 = given
  less than requested; above 1 = given more (e.g. a top-up).
- **`GOODSPRICE_TO_AMT`** — price of the goods ÷ amount asked for.
- **`AMT_ANNUITY_TO_CREDIT_RATIO`** — yearly repayment ÷ amount granted.

Each is left blank when its denominator is 0 or missing.

In [ ]:
PrevLoans['CREDIT_TO_APPL'] = np.where(
    PrevLoans['AMT_APPLICATION'] > 0,
    PrevLoans['AMT_CREDIT'] / PrevLoans['AMT_APPLICATION'],
    np.nan,
)
PrevLoans['GOODSPRICE_TO_AMT'] = np.where(
    PrevLoans['AMT_APPLICATION'] > 0,
    PrevLoans['AMT_GOODS_PRICE'] / PrevLoans['AMT_APPLICATION'],
    np.nan,
)
PrevLoans['AMT_ANNUITY_TO_CREDIT_RATIO'] = np.where(
    PrevLoans['AMT_CREDIT'] > 0,
    PrevLoans['AMT_ANNUITY'] / PrevLoans['AMT_CREDIT'],
    np.nan,
)

print("Ratio features added.")

## 5. Save the Cleaned Loan-Level Table

One row per previous application (`SK_ID_PREV`), with the ratios above. The
credit card, POS and repayment notebooks read this file to add
previous-application details to their own data. It contains no `TARGET`.

In [ ]:
PrevLoans.to_parquet(PATH_OUT_ENRICHED, index=False)
print(f"Saved enriched parquet → {PATH_OUT_ENRICHED}")
print(f"Columns: {PrevLoans.shape[1]} | Rows: {PrevLoans.shape[0]:,}")

## 6. Split into Refused and Not-Refused Applications

- **`ApprovedLoans`** — everything not refused: approved, cancelled by the
  applicant, and unused offers. Cancelled ones are kept because credit was
  still offered. (Not used further below.)
- **`RefusedLoans`** — applications Home Credit turned down.

In [ ]:
ApprovedLoans = PrevLoans[PrevLoans['NAME_CONTRACT_STATUS'] != 'Refused'].copy()
RefusedLoans  = PrevLoans[PrevLoans['NAME_CONTRACT_STATUS'] == 'Refused'].copy()

print(f"ApprovedLoans : {len(ApprovedLoans):,} rows")
print(f"RefusedLoans  : {len(RefusedLoans):,} rows")

## 7. Helper Functions

- `flag_merge()` — joins a table on and adds a yes/no column saying whether
  each applicant had a match (e.g. "had an application in the last year").
- `build_window_agg()` — averages some columns over applications decided in
  a recent period (last 3 years, 1 year, 6 months or 3 months).

In [ ]:
def flag_merge(left, right, on, flag_col):
    """Left-join `right` onto `left` and add `flag_col`: 1 if the row had a match in `right`, else 0."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


def build_window_agg(df, agg_cols, days_back, suffix, prefix):
    """
    Average of each column in `agg_cols`, per applicant, over applications
    decided in the last `days_back` days.

    Parameters
    ----------
    df        : table with DAYS_DECISION and SK_ID_CURR columns
    agg_cols  : columns to average
    days_back : how far back, as a negative number of days (e.g. -365*3 = 3 years)
    suffix    : added to the end of each new column name, e.g. '_LAST3Y'
    prefix    : added to the start, e.g. 'PREVIOUS_TOTAL_LOAN_'

    New columns are named <prefix><column>_MEAN<suffix> in capitals, e.g.
    PREVIOUS_TOTAL_LOAN_CREDIT_TO_APPL_MEAN_LAST1Y.
    """
    window = df[df['DAYS_DECISION'] >= days_back]
    agg = (
        window.groupby('SK_ID_CURR')[agg_cols]
        .agg(['mean'])
        .reset_index()
    )
    # Turn the two-level column names (column, 'mean') into single names
    agg.columns = [
        f"{prefix}{col[0]}_{col[1]}{suffix}".upper() if col[1] else col[0]
        for col in agg.columns
    ]
    return agg

## 8. Summaries Over All Previous Applications

One row per applicant, over every previous application (approved,
cancelled and refused):

- how many applications in total, and in the last 3 years / year / 6 months / 3 months
- average amount granted and goods-to-amount ratio
- `PREVIOUS_TOTAL_LOAN_LAST_DECISION_DAY` — how recent the latest decision
  was (days before this application, negative)
- `PREVIOUS_TOTAL_LOAN_MULTIPLE_ATTEMPTS_COUNT` — applications that weren't
  the last one for their contract (i.e. repeat attempts)
- averages of the four ratio/amount columns over each recent period, plus a
  yes/no flag per period saying whether the applicant had any application in it

In [ ]:
agg_cols = ['AMT_CREDIT', 'GOODSPRICE_TO_AMT', 'CREDIT_TO_APPL', 'AMT_ANNUITY_TO_CREDIT_RATIO']

# DAYS_DECISION is days before this application (negative), so ">= -365"
# means "decided within the last year"
PrevLoans_Aggregate_CurrentID = (
    PrevLoans.groupby('SK_ID_CURR').agg(
        PREVIOUS_TOTAL_LOAN_COUNT=('NAME_CONTRACT_STATUS', 'count'),        # all applications
        PREVIOUS_TOTAL_LOAN_AVG_LOAN_AMOUNT=('AMT_CREDIT', 'mean'),
        PREVIOUS_TOTAL_LOAN_AVG_GOODS_TO_AMT_RATIO=('GOODSPRICE_TO_AMT', 'mean'),
        PREVIOUS_TOTAL_LOAN_LAST3YEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365 * 3).sum()),
        PREVIOUS_TOTAL_LOAN_LASTYEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365).sum()),
        PREVIOUS_TOTAL_LOAN_LAST6MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -180).sum()),
        PREVIOUS_TOTAL_LOAN_LAST3MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -90).sum()),
        # Most recent decision on any application (the max of negative days)
        PREVIOUS_TOTAL_LOAN_LAST_DECISION_DAY=('DAYS_DECISION', 'max'),
        PREVIOUS_TOTAL_LOAN_MULTIPLE_ATTEMPTS_COUNT=(
            'FLAG_LAST_APPL_PER_CONTRACT', lambda x: (x == 'N').sum()
        ),
    )
    .reset_index()  # make SK_ID_CURR a normal column again, for the merges below
)

# Averages over recent periods
PrevLoans_3Y_Agg = build_window_agg(PrevLoans, agg_cols, -365*3, '_LAST3Y', 'PREVIOUS_TOTAL_LOAN_')
PrevLoans_1Y_Agg = build_window_agg(PrevLoans, agg_cols, -365,   '_LAST1Y', 'PREVIOUS_TOTAL_LOAN_')
PrevLoans_6M_Agg = build_window_agg(PrevLoans, agg_cols, -180,   '_LAST6M', 'PREVIOUS_TOTAL_LOAN_')
PrevLoans_3M_Agg = build_window_agg(PrevLoans, agg_cols, -90,    '_LAST3M', 'PREVIOUS_TOTAL_LOAN_')

print(f"All-loan base agg : {PrevLoans_Aggregate_CurrentID.shape}")
print(f"3Y / 1Y / 6M / 3M: {PrevLoans_3Y_Agg.shape} / {PrevLoans_1Y_Agg.shape} "
      f"/ {PrevLoans_6M_Agg.shape} / {PrevLoans_3M_Agg.shape}")

In [ ]:
PrevLoans_Aggregate_final = PrevLoans_Aggregate_CurrentID.copy()

PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_3Y_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_3Y'
)
PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_1Y_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_1Y'
)
PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_6M_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_6M'
)
PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_3M_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_3M'
)

del PrevLoans_Aggregate_CurrentID, PrevLoans_3Y_Agg, PrevLoans_1Y_Agg, PrevLoans_6M_Agg, PrevLoans_3M_Agg
print(f"All-loan aggregate shape: {PrevLoans_Aggregate_final.shape}")

## 9. Summaries Over Refused Applications Only

The same kind of summaries, but only over applications Home Credit refused:
how many, how recently (`PREVIOUS_REFUSED_LOAN_LAST_REFUSED_DAY`), the
amounts involved, averages over recent periods, and a yes/no flag per
period. Applicants never refused get blanks in all of these.

The current @prod model uses `PREVIOUS_REFUSED_LOAN_LAST3YEAR_COUNT` from here.

In [ ]:
RefusedLoans_Aggregate_CurrentID = (
    RefusedLoans.groupby('SK_ID_CURR').agg(
        PREVIOUS_REFUSED_LOAN_COUNT=('NAME_CONTRACT_STATUS', 'count'),      # refused applications
        PREVIOUS_REFUSED_LOAN_AVG_LOAN_AMOUNT=('AMT_CREDIT', 'mean'),
        PREVIOUS_REFUSED_LOAN_AVG_GOODS_TO_AMT_RATIO=('GOODSPRICE_TO_AMT', 'mean'),
        PREVIOUS_REFUSED_LOAN_LAST3YEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365 * 3).sum()),
        PREVIOUS_REFUSED_LOAN_LASTYEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365).sum()),
        PREVIOUS_REFUSED_LOAN_LAST6MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -180).sum()),
        PREVIOUS_REFUSED_LOAN_LAST3MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -90).sum()),
        # Most recent refusal
        PREVIOUS_REFUSED_LOAN_LAST_REFUSED_DAY=('DAYS_DECISION', 'max'),
        PREVIOUS_REFUSED_LOAN_MULTIPLE_ATTEMPTS_COUNT=(
            'FLAG_LAST_APPL_PER_CONTRACT', lambda x: (x == 'N').sum()
        ),
    )
    .reset_index()  # make SK_ID_CURR a normal column again, for the merges below
)

RefusedLoans_3Y_Agg = build_window_agg(RefusedLoans, agg_cols, -365*3, '_LAST3Y', 'PREVIOUS_REFUSED_LOAN_')
RefusedLoans_1Y_Agg = build_window_agg(RefusedLoans, agg_cols, -365,   '_LAST1Y', 'PREVIOUS_REFUSED_LOAN_')
RefusedLoans_6M_Agg = build_window_agg(RefusedLoans, agg_cols, -180,   '_LAST6M', 'PREVIOUS_REFUSED_LOAN_')
RefusedLoans_3M_Agg = build_window_agg(RefusedLoans, agg_cols, -90,    '_LAST3M', 'PREVIOUS_REFUSED_LOAN_')

print(f"Refused-loan base agg: {RefusedLoans_Aggregate_CurrentID.shape}")

In [ ]:
RefusedLoans_Aggregate_final = RefusedLoans_Aggregate_CurrentID.copy()

RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_3Y_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_3Y'
)
RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_1Y_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_1Y'
)
RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_6M_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_6M'
)
RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_3M_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_3M'
)

del RefusedLoans_Aggregate_CurrentID, RefusedLoans_3Y_Agg, RefusedLoans_1Y_Agg, RefusedLoans_6M_Agg, RefusedLoans_3M_Agg
print(f"Refused-loan aggregate shape: {RefusedLoans_Aggregate_final.shape}")

## 10. Combine the Two

Adds the refused-application features to the all-application ones, one row
per applicant. Applicants never refused get blanks in the refused columns.

In [ ]:
PrevLoans_Aggregate_final = PrevLoans_Aggregate_final.merge(
    RefusedLoans_Aggregate_final, on='SK_ID_CURR', how='left'
)

del RefusedLoans_Aggregate_final, ApprovedLoans, RefusedLoans
import gc; gc.collect()

print(f"Final aggregate shape: {PrevLoans_Aggregate_final.shape}")
PrevLoans_Aggregate_final.head(3)

## 11. Quality Checks

Exactly one row per applicant, no `TARGET` column, and the share of missing
values in the emptiest columns.

In [ ]:
# No duplicate customers
n_dupes = PrevLoans_Aggregate_final['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR found: {n_dupes}"
print("No duplicate SK_ID_CURR.")

# TARGET must not be present
assert 'TARGET' not in PrevLoans_Aggregate_final.columns, "TARGET leaked into output!"
print("TARGET column not present — no data leakage.")

# Null rates
null_rates = PrevLoans_Aggregate_final.isna().mean().sort_values(ascending=False)
print(f"\nNull rate (top 10 columns):")
print(null_rates.head(10).to_string())

print(f"\nFinal row count: {len(PrevLoans_Aggregate_final):,}")

## 12. Save

In [ ]:
PrevLoans_Aggregate_final.to_parquet(PATH_OUT_AGG, index=False)
print(f"Saved → {PATH_OUT_AGG}")

## 13. Free Memory

In [ ]:
del PrevLoans, PrevLoans_Aggregate_final, base
import gc; gc.collect()
print("All DataFrames released.")